### Step 1: Environment Setup 

In [0]:
# ==============================================================================
# Cell 1: Environment Setup & Library Installation
# ==============================================================================
%pip install -U langchain langchain-openai langgraph tavily-python 
dbutils.library.restartPython()

**Package metadata verifications**

In [0]:
import importlib.metadata

package_name = "tavily-python"
package_name = "langchain"
package_name = "langchain-openai"
package_name = "langgraph"
#package_name = "beautifulsoup4"

try:
    # Check if package metadata exists
    version = importlib.metadata.version(package_name)
    print(f"'{package_name}' is already installed (version {version}). Skipping installation.")
except importlib.metadata.PackageNotFoundError:
    print(f"'{package_name}' is NOT installed. Proceeding to install...")
    # Place your installation command here, e.g.:
    # %pip install tavily-python

### Databricks Secrets 

Databricks secret management keeps credentials out of your code by grouping secrets inside secret scopes (in a workspace or in Unity Catalog) and enforcing access permissions. You create a scope and store key-value secrets via the CLI or API

**Bash Scripts**

databricks secrets create-scope client-research

databricks secrets put-secret client-research tavily-api-key --string-value "your-secret-value"

In [0]:
# ==============================================================================
# Secure Configuration via Databricks Secrets & Native Databricks LLM
# ==============================================================================
import os

# Safely retrieve the Tavily API key from your Databricks secret scope
try:
    os.environ["TAVILY_API_KEY"] = dbutils.secrets.get(
        scope="client-research",
        key="tavily-api-key",
    )
    print("Successfully loaded Tavily API key from Databricks Secrets.")
except Exception as e:
    print(f"Error loading secret: {e}. Please ensure the secret scope and key exist.")

### Creating a Dynamic UI

In [0]:
# Create a dynamic UI widget at the top of the Databricks notebook for interactive testing
dbutils.widgets.text("company_name", "Snowflake", "Target Company Name")
target_company = dbutils.widgets.get("company_name")
print(f"Target Company configured via UI Widget: {target_company}")

### Define a Search Tool and LangGraph Agent 

In [0]:
# ==============================================================================
# Cell 3: Define Public Web Research Tool & LangGraph ReAct Agent
# ==============================================================================
# databricks-langchain's dependency tree resolves to langgraph 1.0.10, but leaves the
# newer langgraph-prebuilt installed — whose tool_node imports symbols missing from
# that runtime. Pin prebuilt to the version range floor that matches it.
%pip install databricks-langchain "langgraph-prebuilt==1.0.8"

from tavily import TavilyClient
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from databricks_langchain import ChatDatabricks
# @tool decorator & search_public_web function: 
@tool
def search_public_web(query: str) -> str:
    """
    Searches the public web for company profile, revenue, cloud modernization, 
    AI/data investments, leadership news, and open Gartner industry insights.
    """
    tavily = TavilyClient(api_key=dbutils.secrets.get(
        scope="client-research",
        key="tavily-api-key",
    ))
    response = tavily.search(query=query, max_results=5, search_depth="advanced")
    
    formatted_results = []
    for r in response.get("results", []):
        formatted_results.append(f"Title: {r['title']}\nURL: {r['url']}\nSnippet: {r['content']}\n---")
    return "\n".join(formatted_results) if formatted_results else "No public information found."

# Initialize Foundation Model and Toolset
# llm = ChatOpenAI(model="gpt-4o", temperature=0.1)
tools = [search_public_web]

# Initialize the native Databricks Foundation Model endpoint
llm = ChatDatabricks(
    endpoint="databricks-meta-llama-3-3-70b-instruct",  # Swap to your workspace's active endpoint if needed
    temperature=0.1 # setting keeps its answers focused, consistent, and less "creative" or imaginative—which is what you want for research
)

# Strict system prompt ensuring factual grounding and distinct recommendation separation
SYSTEM_PROMPT = """
You are an Enterprise Account Research & Qualification Agent built on Databricks.
Your task is to research the requested company using ONLY the `search_public_web` tool and evaluate it against five specific criteria.

MANDATORY RESEARCH STRATEGY:
Execute multiple searches covering:
1. Company Size & Scale (revenue, headcount, market presence).
2. Technology Modernization (cloud, ERP/CRM, digital initiatives).
3. AI & Data Focus (AI, GenAI, analytics, data platform investments).
4. Relevant Industry Trends & Open Gartner Insights (search public Gartner press releases or research summaries).
5. Near-Term Opportunity Signals (recent expansions, leadership changes, investments, challenges).

STRICT GUARDRAILS:
- Never fabricate information or estimate missing metrics. If public evidence is missing, explicitly state "Not enough public evidence found."
- Every factual claim MUST include an inline markdown link citation [Source Title](URL).
- Clearly separate SOURCED FACTS from AI-GENERATED RECOMMENDATIONS.

REQUIRED OUTPUT FORMAT (Markdown):
# Client Brief: [Company Name]

## 1. Company Overview (Sourced Facts)
- Provide a summary of the business model, scale, and market position based strictly on search results.

## 2. Five Criteria Assessment (Sourced Facts & Citations)
- **Company Size & Scale:** [Findings + Source Link]
- **Technology Modernization:** [Findings + Source Link]
- **AI & Data Focus:** [Findings + Source Link]
- **Relevant Industry Trends & Open Gartner Insights:** [Findings + Source Link]
- **Near-Term Opportunity Signals:** [Findings + Source Link]

## 3. Overall Prospect Fit (AI-Generated Recommendation)
- **Classification:** [Good Fit / Potential Fit / Not Enough Evidence]
- **Rationale:** [Justification tied strictly to Section 2 findings]

## 4. Key Technology Priorities & Opportunities (AI-Generated Recommendation)
- Bulleted strategic opportunities for engagement.

## 5. Initial Meeting Prep (AI-Generated Recommendation)
- **5 Tailored Discovery Questions**
- **Key Talking Points**
"""

agent = create_react_agent(llm, tools=tools, prompt=SYSTEM_PROMPT)

In [0]:
# ==============================================================================
# Cell 4: Execute Agent & Render Formatted Client Brief
# ==============================================================================
# Refresh a stale cached langchain_core.tracers.context (left over from a mid-session
# package downgrade) — the installed langchain-core defines _get_tracer_kwargs,
# which agent.invoke() requires
import importlib
import langchain_core.tracers.context
importlib.reload(langchain_core.tracers.context)

target_company = dbutils.widgets.get("company_name")
print(f"Executing agent research workflow for: {target_company}...\n" + "="*60)

response = agent.invoke({
    "messages": [("user", f"Research {target_company} and generate the comprehensive client brief according to the required format.")]
})

final_report = response["messages"][-1].content

# Render the formatted Markdown report directly within the notebook output
displayHTML(final_report.replace("\n", "<br>"))

### Formatted output

In [0]:
from IPython.display import display, Markdown

target_company = dbutils.widgets.get("company_name")
print(f"Executing agent research workflow for: {target_company}...\n" + "="*60)

response = agent.invoke({
    "messages": [("user", f"Research {target_company} and generate the comprehensive client brief according to the required format.")]
})

final_report = response["messages"][-1].content

# Render clean, professional Markdown output natively in Databricks
display(Markdown(final_report))